<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.5.3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# **MNPS Job Classification_Kimi (Two-Pass Self-Consistency)**
> A notebook to help you get started  
> DSI DSSG + MNPS   
> # **Version 7.5.4 Changes (Modified for Kimi)**
> - **Two-Pass Classification**:
>   - Pass 1: Initial LLM classification (attribute-only, full context)
>   - Pass 2: Self-consistency check — LLM reviews its own justification vs classification and corrects mismatches
> - **Preserves all v7.5.3 logic**: Drive mounting, prompts, model, post-processing, validation
> - **Addresses Justification Mismatch Problem** without changing output format
> - **Still ignores original job title**, uses full MNPS role/competency context

# **Section 1: Setup, Imports, Mount Drive, Load Data**


In [ ]:
# ==== CELL 1: Imports and Setup ====
import os, json, zipfile, re, time, random
from pathlib import Path
from datetime import datetime
from typing import Dict, Tuple

import pandas as pd
import numpy as np
from tqdm import tqdm
from google.colab import drive, userdata
from openai import OpenAI

# Mount Drive
drive.mount('/content/drive')

# Create run folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = Path('/content')
OUTPUTS_DIR = Path(f"/content/drive/My Drive/Colab Notebooks/Run Results/RUN_{timestamp}/outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Outputs directory: {OUTPUTS_DIR}")

# **Section 2: Kimi API Configuration**


In [ ]:
# ==== CELL 2: Kimi LLM Client Configuration ====
# Get API key from Colab secrets panel
KIMI_API_KEY = userdata.get("KIMI_API_KEY")

# Initialize Kimi client
client = OpenAI(
    api_key=KIMI_API_KEY,
    base_url="https://api.moonshot.cn/v1"
)

# Model selection
MODEL_ID = "moonshot-v1-32k"  # Use 32k context for full KSACs
print(f"✅ Kimi client initialized with model: {MODEL_ID}")

# **Section 3: Comprehensive Data Loading**

In [ ]:
# ==== CELL 3: Robust Data Loading with Zip Extraction ====
# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - please upload it to /content/")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# If main input file not found, check inside the zip
if not BATCH_INPUT_CSV.exists() and ZIP_FILE.exists():
    print("⚠️  Sample JDs.csv not found in root — checking inside MNPS Prompt Resources.zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        if "Sample JDs.csv" in zip_contents:
            zip_ref.extract("Sample JDs.csv", RUN_ROOT)
            print("✅ Extracted Sample JDs.csv from zip")
        else:
            print("❌ Sample JDs.csv not found in zip contents:", zip_contents)
            raise FileNotFoundError("Sample JDs.csv not found in root or zip")

    # Optionally extract Ground Truth if present
    if "Ground Truth Masterfile.csv" in zip_contents:
        zip_ref.extract("Ground Truth Masterfile.csv", RUN_ROOT)
        print("✅ Extracted Ground Truth Masterfile.csv from zip")

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

# Verify all files exist
required_files = {
    "Sample JDs": BATCH_INPUT_CSV,
    "MNPS Roles": MNPS_ROLES_CSV,
    "MNPS KSACs": MNPS_KSACS_CSV,
    "Competencies": COMPETENCY_EXTENDED_CSV,
    "Korn Ferry": KORN_FERRY_CSV
}

print(f"\n{'='*60}")
print("📄 FILE VERIFICATION")
print("="*60)
for name, path in required_files.items():
    if not path.exists():
        print(f"❌ MISSING: {name} at {path}")
    else:
        print(f"✅ Found: {path.name}")

# Load all datasets
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1') if GT_MASTERFILE_CSV.exists() else None
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"\n{'='*60}")
print("📊 DATASET SUMMARY")
print("="*60)
print(f"✅ Job descriptions: {len(df)}")
if gt_df is not None:
    print(f"✅ Ground truth records: {len(gt_df)}")
print(f"✅ MNPS roles: {len(roles_df)}")
print(f"✅ MNPS KSACs: {len(ksacs_df)}")
print(f"✅ Competency descriptions: {len(competency_df)}")
print(f"✅ Korn Ferry competencies: {len(korn_ferry_df)}")
print(f"{'='*60}\n")

# **Section 4: Build Comprehensive KSACs Text**


In [ ]:
# ==== CELL 4: Build Comprehensive KSACs Text ====
def build_ksacs_text() -> str:
    """Construct full KSACs context for Kimi."""
    ksacs_text = "MNPS KNOWLEDGE, SKILLS, ABILITIES, AND COMPETENCIES:\n\n"

    # Role-specific KSACs
    role_col = next((c for c in ksacs_df.columns if 'role' in c.lower()), None)
    ksacs_col = next((c for c in ksacs_df.columns if 'ksacs' in c.lower()), None)
    if role_col and ksacs_col:
        for _, row in ksacs_df.iterrows():
            if pd.notna(row.get(role_col)) and pd.notna(row.get(ksacs_col)):
                ksacs_text += f"**{row[role_col]}**:\n{row[ksacs_col]}\n\n"

    # Competency descriptions
    comp_col = next((c for c in competency_df.columns if 'competency' in c.lower()), None)
    desc_col = next((c for c in competency_df.columns if 'description' in c.lower()), None)
    if comp_col and desc_col:
        ksacs_text += "**COMPETENCY EXTENDED DESCRIPTIONS**:\n"
        for _, row in competency_df.iterrows():
            if pd.notna(row.get(comp_col)) and pd.notna(row.get(desc_col)):
                ksacs_text += f"- {row[comp_col]}: {row[desc_col]}\n"

    # Korn Ferry competencies
    kf_comp = next((c for c in korn_ferry_df.columns if 'competency' in c.lower()), None)
    kf_desc = next((c for c in korn_ferry_df.columns if 'definition' in c.lower() or 'description' in c.lower()), None)
    if kf_comp and kf_desc:
        ksacs_text += "\n**KORN FERRY 38 COMPETENCIES**:\n"
        for _, row in korn_ferry_df.iterrows():
            if pd.notna(row.get(kf_comp)) and pd.notna(row.get(kf_desc)):
                ksacs_text += f"- {row[kf_comp]}: {row[kf_desc]}\n"

    return ksacs_text

KSACS_TEXT = build_ksacs_text()
print(f"✅ Built KSACs text ({len(KSACS_TEXT)} characters)")

# **Section 5: Problem Role Cheat Sheet & Constants**


In [ ]:
# ==== CELL 5: Enhanced Classification Rules ====
VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()

# Roles that should NOT have minor sub-grouping
EXEMPT_FROM_MINOR = {
    'Teacher', 'Principal', 'Assistant Principal', 'Counselor',
    'Librarian', 'Therapist', 'Social Worker', 'Instructor'
}

# Executive roles (rarely "Lead")
EXECUTIVE_ROLES = {'Coordinator', 'Principal', 'Director', 'Manager'}

# Minor role mapping
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

# Specialist fallback patterns
SPECIALIST_PATTERNS = [
    ('Technician', r'technical|repair|maintenance|install|troubleshoot|equipment|hands-on|tools'),
    ('Analyst', r'analyz|data research|evaluat|statistical|metrics|reports'),
    ('Teacher', r'classroom|curriculum|instruction|students'),
    ('Coach', r'instructional coach|mentor|professional development|co-teach'),
    ('Accountant', r'accounting|financial|audit|budget|fiscal'),
    ('Coordinator', r'coordinate|organize|facilitate|liaison'),
]

print("✅ Classification rules loaded")

# **Section 6: Post-Processing Functions**

In [ ]:
# ==== CELL 6: Enhanced Post-Processing for Kimi ====
def enforce_role_justification_alignment(text: str, major: str, justification: str) -> str:
    """Kimi-specific: Ensure classification matches justification keywords."""
    just_lower = justification.lower()
    major_lower = major.lower()

    # If justification doesn't mention the role, reclassify
    if major_lower not in just_lower and major != 'Other':
        if 'coordinate' in just_lower and 'supervis' not in just_lower:
            return 'Coordinator'
        elif 'coach' in just_lower or 'mentor' in just_lower:
            return 'Coach'
        elif 'manage' in just_lower and 'staff' in just_lower:
            return 'Manager'

    return major

def auto_elevate_by_credentials(row: pd.Series, major: str, minor: str) -> Tuple[str, str]:
    """Auto-elevate based on education and experience."""
    education = str(row.get('Education', '')).lower()
    experience = str(row.get('Work Experience', '')).lower()

    # Doctoral degree → Level III
    if 'doctoral' in education or 'phd' in education or 'doctorate' in education:
        return major, 'III'

    # Master's + 5+ years leadership → Level III
    if 'master' in education and '5' in experience:
        if any(exec_role in str(row.get('Job Title', '')).lower() for exec_role in ['director', 'principal', 'executive']):
            return major, 'III'

    return major, minor

def normalize_minor(minor: str) -> str:
    """Normalize minor sub-group."""
    if pd.isna(minor) or minor == '':
        return ''
    s = str(minor).strip()
    if s in ['I', 'II', 'III', 'Lead']:
        return s
    return CANON_MINOR_MAP.get(s.lower(), 'I')

def apply_minor_exemption(major: str, minor: str) -> str:
    """Remove minor grouping for exempt roles."""
    return '' if major in EXEMPT_FROM_MINOR else minor

def fix_executive_minor(major: str, minor: str) -> str:
    """Prevent 'Lead' for executive roles."""
    if major in EXECUTIVE_ROLES and minor == 'Lead':
        return 'II' if major in ['Director', 'Principal'] else 'II'
    return minor

def get_job_text(row: pd.Series) -> str:
    """Extract full job text from row."""
    cols = ['Position Summary', 'Essential Functions', 'Work Experience',
            'Education', 'Licenses and Certifications', 'Knowledge, Skills and Abilities']
    return ' '.join([str(row.get(col, '')) for col in cols])

print("✅ Post-processing functions loaded")

# **Section 7: Enhanced Prompts for Kimi**


In [ ]:
# ==== CELL 7: Kimi-Optimized Prompts ====
PROBLEM_ROLE_CLARIFICATIONS = """
**CRITICAL CLASSIFICATION RULES FOR KIMI:**

1. **ROLE DISTINCTION - Coordinator vs Manager:**
   - **Coordinator**: Facilitates, organizes, liaises, manages PROGRAMS (NO hire/fire authority)
   - **Manager**: Has AUTHORITY to hire/fire staff, control budgets, develop POLICIES
   - DECISION: If justification mentions "coordinate" ≥3 times but NOT "supervise staff" → Coordinator

2. **EXEMPT ROLES - No Minor Sub-Grouping:**
   Teacher, Principal, Assistant Principal, Counselor, Librarian, Therapist, Social Worker, Instructor
   → minor_sub_group MUST be empty string ""

3. **AUTO-ELEVATION TRIGGERS:**
   - Doctoral degree requirement → "III"
   - "Lead" in original title + supervises staff → "Lead"
   - Master's + 5+ years leadership → "III"

4. **SELF-CHECK BEFORE OUTPUT:**
   - Does my major_role_group appear VERBATIM in my justification?
   - If not, reclassify based on justification's primary verb
"""

ZERO_SHOT_PROMPT = f"""
Objective: Classify MNPS jobs by function attributes only, ignoring titles.

Process:
- Analyze: Education, Experience, Essential Functions, KSACs
- Compare against MNPS standards and competency frameworks
- Provide detailed justification aligned to selected role

{PROBLEM_ROLE_CLARIFICATIONS}

**MNPS Resources:**
{{ksacs_text}}

**Available Roles:** {{valid_roles}}

**Job to Classify:**
{{job_text}}

**OUTPUT AS JSON:**
{{
  "new_job_title": "RoleName Level",
  "major_role_group": "One of available roles",
  "minor_sub_group": "I, II, III, Lead, or blank",
  "grouping_justification": "Detailed explanation matching role"
}}
"""

SELF_CONSISTENCY_PROMPT = """
You are Kimi, an expert at logical consistency verification.

**Previous Classification:**
{pass1_output}

**VERIFY THESE RULES:**
1. Keyword Match: Does major_role_group appear in grouping_justification?
2. Coordination Check: If "coordinate" appears 3+ times, is major_role_group = "Coordinator"?
3. Supervision Check: If "hire/fire/supervise staff" appears, is major_role_group = "Manager"?
4. Exemption Check: Should minor_sub_group be blank for this role?

**If ANY rule fails:**
- Return CORRECTED JSON with original justification unchanged
- Add "correction_reason" field explaining fix

**If ALL pass:**
- Return original JSON unchanged
"""

# **Section 8: Two-Pass Processing Loop**


In [ ]:
# ==== CELL 8: Main Processing Function ====
def process_job_with_kimi(row_idx: int, row: pd.Series) -> dict:
    """Two-pass Kimi classification with enhanced post-processing."""

    job_text = get_job_text(row)

    # === PASS 1: Initial Classification ===
    pass1_prompt = ZERO_SHOT_PROMPT.format(
        ksacs_text=KSACS_TEXT[:15000],  # Use first 15k chars to fit context
        valid_roles=', '.join(VALID_ROLES),
        job_text=job_text
    )

    try:
        pass1_result = call_kimi_json_with_retry(pass1_prompt)

        # Extract values
        major = pass1_result.get('major_role_group', 'Other')
        minor = pass1_result.get('minor_sub_group', 'I')
        justification = pass1_result.get('grouping_justification', '')

        # === PYTHON PRE-CORRECTIONS (before Pass 2) ===
        # Apply specialist fallback
        if major == 'Specialist':
            for role_name, pattern in SPECIALIST_PATTERNS:
                if re.search(pattern, job_text, re.I):
                    major = role_name
                    break

        # === PASS 2: Self-Consistency Check ===
        pass2_prompt = SELF_CONSISTENCY_PROMPT.format(
            pass1_output=json.dumps(pass1_result, indent=2)
        )

        pass2_result = call_kimi_json_with_retry(pass2_prompt)

        # === ENHANCED POST-PROCESSING ===
        final_major = pass2_result.get('major_role_group', major)
        final_minor = pass2_result.get('minor_sub_group', minor)

        # 1. Enforce justification alignment
        final_major = enforce_role_justification_alignment(
            job_text, final_major, justification
        )

        # 2. Auto-elevate by credentials
        final_major, final_minor = auto_elevate_by_credentials(
            row, final_major, final_minor
        )

        # 3. Apply minor exemptions
        final_minor = apply_minor_exemption(final_major, final_minor)

        # 4. Fix executive minor grouping
        final_minor = fix_executive_minor(final_major, final_minor)

        # 5. Normalize
        final_minor = normalize_minor(final_minor)

        # Build final title
        final_title = f"{final_major} {final_minor}".strip() if final_minor else final_major

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': final_title,
            'major_role_group': final_major,
            'minor_sub_group': final_minor,
            'grouping_justification': justification,
            'model_used': MODEL_ID,
            'pass1_raw': major,
            'corrections_applied': final_major != major or final_minor != minor
        }

    except Exception as e:
        print(f"❌ Error on row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'ERROR',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'ERROR: {str(e)}',
            'model_used': MODEL_ID,
            'pass1_raw': 'ERROR',
            'corrections_applied': False
        }

# **Section 9: Execute Batch Processing**


In [ ]:
# ==== CELL 9: Run Classifications ====
results = []
corrections = []

print(f"🚀 Processing {len(df)} jobs with Kimi...")

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying"):
    result = process_job_with_kimi(idx, row)
    results.append(result)

    if result['corrections_applied']:
        corrections.append({
            'row': idx,
            'title': result['job_title_original'],
            'corrected_to': result['major_role_group'],
            'reason': 'Post-processing correction'
        })

    time.sleep(0.3)  # Rate limit protection

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Kimi_v1.csv"
results_df.to_csv(output_path, index=False)

# Save corrections log
if corrections:
    pd.DataFrame(corrections).to_csv(OUTPUTS_DIR / "corrections_log.csv", index=False)

print(f"✅ Complete! Saved to: {output_path}")
print(f"✅ Corrections applied: {len(corrections)}")

# **Section 10: Validation & Summary**

In [ ]:
# ==== CELL 10: Generate Summary Statistics ====
def generate_summary(preds_df: pd.DataFrame):
    """Create summary report of classifications."""

    stats = {
        'total_jobs': len(preds_df),
        'unique_roles': preds_df['major_role_group'].nunique(),
        'minor_distribution': preds_df['minor_sub_group'].value_counts().to_dict(),
        'corrections_applied': preds_df['corrections_applied'].sum(),
        'errors': (preds_df['major_role_group'] == 'ERROR').sum(),
        'specialist_count': (preds_df['major_role_group'] == 'Specialist').sum(),
        'exempt_with_minor': sum(
            (preds_df['major_role_group'].isin(EXEMPT_FROM_MINOR)) &
            (preds_df['minor_sub_group'] != '')
        )
    }

    # Save detailed stats
    stats_path = OUTPUTS_DIR / "summary_statistics.json"
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)

    # Print summary
    print("\n" + "="*50)
    print("📊 CLASSIFICATION SUMMARY")
    print("="*50)
    print(f"Total Jobs Processed: {stats['total_jobs']}")
    print(f"Unique Major Roles: {stats['unique_roles']}")
    print(f"Corrections Applied: {stats['corrections_applied']}")
    print(f"Errors: {stats['errors']}")
    print(f"\nMinor Role Distribution:")
    for k, v in stats['minor_distribution'].items():
        print(f"  {k}: {v}")

    print(f"\nExempt roles with minor grouping (SHOULD BE 0): {stats['exempt_with_minor']}")
    print(f"Specialist count (should be minimal): {stats['specialist_count']}")

    return stats

summary = generate_summary(results_df)

# **Section 11: Interactive Debugging (Optional)**

In [ ]:
# ==== CELL 11: Debug a Specific Classification ====
def debug_classification(row_idx: int, preds_df: pd.DataFrame, original_df: pd.DataFrame):
    """Interactive debugging for a specific row."""

    row = original_df.iloc[row_idx]
    pred = preds_df.iloc[row_idx]

    debug_prompt = f"""
    **DEBUG SESSION - Row {row_idx}**

    Job: {row['Job Description Name']}

    Text: {get_job_text(row)[:500]}...

    Kimi Classified As: {pred['major_role_group']} - {pred['minor_sub_group']}

    Justification Snippet: {pred['grouping_justification'][:200]}...

    **ANALYSIS QUESTIONS:**
    1. What is the PRIMARY verb in the justification? (coordinate/manage/analyze/etc)
    2. What is the PRIMARY noun in the justification? (programs/staff/data/etc)
    3. Based on these, is the classification correct? If not, what should it be?
    4. What specific change to the prompt would prevent this error?

    Return JSON with keys: primary_verb, primary_noun, is_correct, correct_classification, prompt_fix
    """

    result = call_kimi_json_with_retry(debug_prompt)
    print(json.dumps(result, indent=2))

    return result

# Example usage - change index to debug specific job
# debug_result = debug_classification(14, results_df, df)

# **Section 12: Export for Comparison**

In [ ]:
# ==== CELL 12: Export Results for Comparison ====
# Merge with original data for side-by-side analysis
export_df = df.copy()
export_df = export_df.merge(
    results_df[['source_row_index', 'new_job_title', 'major_role_group', 'minor_sub_group', 'grouping_justification']],
    left_index=True,
    right_on='source_row_index',
    how='left'
)

# Save detailed results
comparison_path = OUTPUTS_DIR / "detailed_classifications_with_justifications.csv"
export_df.to_csv(comparison_path, index=False)

# Save sample of 10 for quick review
sample_path = OUTPUTS_DIR / "sample_classifications.csv"
export_df.head(10).to_csv(sample_path, index=False)

print(f"✅ Detailed results: {comparison_path}")
print(f"✅ Sample file: {sample_path}")